# الدرس 08 - نمط تصميم الوكيل المتعدد


## الإعداد


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

import os
import asyncio
import dotenv

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## لماذا أنظمة متعددة الوكلاء؟

تتطلب المهام الواقعية مثل تخطيط الرحلات العديد من أنواع الخبرات المختلفة — اللوجستيات، المعرفة المحلية، الميزانية، وأكثر. يصبح الوكيل الواحد الذي يحاول التعامل مع كل شيء سريعًا غير عملي.

تحل أنظمة الوكلاء المتعددة هذا من خلال **التخصص**: يركز كل وكيل على مجال خبرة واحد، مما ينتج نتائج ذات جودة أعلى من العام. كما أنها تحسن **قابلية التوسع** — يمكنك إضافة وكلاء جدد (مثل متخصص في الرحلات الجوية، ناقد مطاعم) دون إعادة كتابة سير العمل الحالي. يتم تجميع الوكلاء معًا من خلال خط أنابيب منظم، حيث يتم تمرير السياق من واحد إلى الآخر.


## إنشاء وكلاء متخصصين


In [ ]:
planner_agent = client.as_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = client.as_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## بناء سير عمل متسلسل

يتيح لك `WorkflowBuilder` توصيل الوكلاء في رسم بياني موجه. هنا نقوم بإنشاء خط أنابيب بسيط من خطوتين: يقوم **منسق السفر** بصياغة جدول الرحلة، ثم يقوم **مساعد السفر** بمراجعته وتحسينه.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## إضافة المزيد من الوكلاء إلى سير العمل

واحدة من أكبر مزايا نمط الوكيل المتعدد هي سهولة التوسعة. أدناه نضيف وكيل **مراجع الميزانية** الذي يتحقق من الخطة مقابل ميزانية المسافر، ويعلم عن العناصر التي قد تدفع التكاليف لتتجاوز الحد، ويقترح بدائل لتوفير المال. الآن يعمل سير العمل مع ثلاثة وكلاء بالتسلسل:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = client.as_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## ملخص

في هذا الدرس تعلمت كيف:

1. **إنشاء وكلاء متخصصين** — كلٌ له دور محدد (التخطيط، الكونسيرج، مراجعة الميزانية).
2. **ربط الوكلاء في سير عمل تسلسلي** باستخدام `WorkflowBuilder` و `add_edge`.
3. **بث الإخراج** من خط أنابيب متعدد الوكلاء، مع تتبع أي وكيل يتحدث.
4. **تمديد سير العمل** عن طريق إضافة وكلاء جدد إلى السلسلة دون تعديل الوكلاء الحاليين.

نمط التصميم متعدد الوكلاء يجعل كل وكيل بسيطًا وفي الوقت نفسه ينتج نتائج أغنى وأكثر مراجعةً بدقة مما يمكن لوكيل واحد تحقيقه بمفرده.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**تنويه**:
تمت ترجمة هذا المستند باستخدام خدمة الترجمة بالذكاء الاصطناعي [Co-op Translator](https://github.com/Azure/co-op-translator). بينما نسعى للدقة، يرجى العلم أن الترجمات الآلية قد تحتوي على أخطاء أو عدم دقة. يجب اعتبار المستند الأصلي بلغته الأصلية المصدر الرسمي والمعتمد. للمعلومات الهامة، يُنصح بالاستعانة بترجمة بشرية محترفة. نحن غير مسؤولين عن أي سوء فهم أو تفسير ناتج عن استخدام هذه الترجمة.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
